# Genesis / Colab 06 — クラッシュ原因の最小切り分け

目的は **06本体の再実行ではなく「落ちた処理の特定」**です。Notebook 04で用意した `HFH_READ` や学習済みモデルは不要です。トークンを入力しないでください。

**前提：** GPUランタイムを選択してから「ランタイムを接続」。06を動かした同じセッションではなく、可能なら新しいランタイムで実行します。最初から長い3D動画を出力しません。

この診断Notebookは `torch` → `Genesis import` → `Genesis init` → `床だけ1ステップ` → `96×72画像1枚` を **別Pythonプロセス** で順番に調べ、最初に失敗した段階で止まります。ログと終了コードをZIPにします。

**注意：** Colab VMそのものが終了した場合はZIPを出力できません。GPU/ドライバ問題、OOM、Segmentation faultのいずれかを予断せず切り分けます。

## A. 実行環境の収集（重いライブラリはNotebook本体ではimportしません）

In [1]:
import sys
import os
import platform
import pathlib
import subprocess
import json
import shutil
import time
from pathlib import Path
OUT = Path('/content/genesis06_diagnostics') if Path('/content').exists() else Path.cwd()/'genesis06_diagnostics'
OUT.mkdir(exist_ok = True, parents = True)
cmds = {'python':[sys.executable,'--version'],'gpu':['nvidia-smi'],'genesis-package':[sys.executable,'-m','pip','show','genesis-world'],'torch-package':[sys.executable,'-m','pip','show','torch']}
for label, cmd in cmds.items():
    try:
        result = subprocess.run(cmd, stdout = subprocess.PIPE, stderr = subprocess.STDOUT, timeout = 30, text = True)
        (OUT/f'env_{label}.txt').write_text(f'command={cmd[0]} (args omitted)\nrc={result.returncode}\n{result.stdout[:15000]}',encoding='utf8')
        print(label, 'returncode=', result.returncode, result.stdout[:500].replace('\n',' | '))
    except Exception as exc:
        print(label, type(exc).__name__, str(exc))
print('kernel-python:', sys.version)
print('logs:', OUT)

python returncode= 0 Python 3.13.16 | 
gpu FileNotFoundError [Errno 2] No such file or directory: 'nvidia-smi'
genesis-package returncode= 1 WARNING: Package(s) not found: genesis-world | 
torch-package returncode= 0 Name: torch | Version: 2.11.0+cpu | Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration | Home-page: https://pytorch.org | Author:  | Author-email: PyTorch Team <packages@pytorch.org> | License: BSD-3-Clause | Location: /usr/local/lib/python3.13/dist-packages | Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions | Required-by: accelerate, fastai, peft, sentence-transformers, timm, torchdata, torchvision | 
kernel-python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
logs: /content/genesis06_diagnostics


## B. Genesisが未インストールの場合のみインストール

上の `genesis-package` が `returncode=1` の場合に限り、次のセルの先頭コメントを外して実行してください。既にインストール済みならスキップします。Pythonライブラリ更新後はランタイム再起動が必要になる場合があります。

In [2]:
# 初期状態ではインストールを行いません。
# 既に以前のNotebookでインストール済みなら、このセルを実行せず次に進んでください。
# %pip install 'genesis-world==1.4.4'
print('Genesisが未導入の場合だけ、コメントを外してインストールしてください。')

Genesisが未導入の場合だけ、コメントを外してインストールしてください。


## C. 独立プロセスで順次検査（クラッシュする直前の印を残します）

In [3]:
# probe06.py をColab内に保存（Notebook単体で完結します）。
from pathlib import Path
PROBE = Path('probe06.py')
PROBE.write_text('#!/usr/bin/env python3\n"""Diagnostic child process, no HF_TOKEN and no user data are accessed."""\nimport argparse,sys,os,time,json,traceback,faulthandler\nfaulthandler.enable(all_threads=True)\n\ndef stage(name, **details):\n    print(\'CHECKPOINT \'+json.dumps({\'step\':name,\'ts\':time.time(),**details},default=str),flush=True)\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--test\', choices=[\'torch\',\'genesis_import\',\'genesis_init\',\'physics\',\'render\'],required=True)\n    p.add_argument(\'--backend\', choices=[\'cpu\',\'cuda\'],default=\'cuda\')\n    p.add_argument(\'--gl\', choices=[\'egl\',\'osmesa\'],default=\'egl\')\n    p.add_argument(\'--output\',default=\'outputs\')\n    a=p.parse_args()\n    os.environ[\'PYOPENGL_PLATFORM\']=a.gl\n    os.environ[\'PYGLET_HEADLESS\']=\'true\'\n    stage(\'entry\',test=a.test,backend=a.backend,gl=a.gl,python=sys.version)\n    if a.test==\'torch\':\n        import torch\n        stage(\'torch_imported\',version=torch.__version__,cuda=torch.cuda.is_available(),torch_cuda_version=torch.version.cuda)\n        if a.backend==\'cuda\':\n            if not torch.cuda.is_available():raise RuntimeError(\'torch.cuda.is_available=False\')\n            x=torch.ones((16,16),device=\'cuda\');stage(\'torch_gpu_basic_math\',value=float((x@x)[0,0].item()),gpu=torch.cuda.get_device_name(0))\n        stage(\'pass\');return\n    stage(\'genesis_import_start\')\n    import genesis as gs\n    stage(\'genesis_import_ok\',version=getattr(gs,\'__version__\',\'unknown\'))\n    if a.test==\'genesis_import\':stage(\'pass\');return\n    import torch\n    if a.backend==\'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA not available from PyTorch\')\n    stage(\'genesis_init_start\')\n    gs.init(backend=gs.cuda if a.backend==\'cuda\' else gs.cpu, logging_level=\'warning\',seed=42)\n    stage(\'genesis_init_ok\')\n    if a.test==\'genesis_init\':stage(\'pass\');return\n    stage(\'scene_prepare\')\n    scene=gs.Scene(show_viewer=False, sim_options=gs.options.SimOptions(dt=0.05))\n    scene.add_entity(gs.morphs.Plane())\n    camera=None\n    if a.test==\'render\':\n        stage(\'add_camera_start\')\n        camera=scene.add_camera(res=(96,72),pos=(2,-2,2),lookat=(0,0,0),GUI=False)\n        stage(\'add_camera_ok\')\n    stage(\'scene_build_start\')\n    scene.build()\n    stage(\'scene_build_ok\')\n    stage(\'scene_step_start\')\n    scene.step()\n    stage(\'scene_step_ok\')\n    if a.test==\'render\':\n        stage(\'camera_render_start\')\n        rgba=camera.render()\n        arr=rgba[0]\n        if arr is None:raise RuntimeError(\'Camera returned no RGB image\')\n        import numpy as np\n        from PIL import Image\n        from pathlib import Path\n        path=Path(a.output);path.mkdir(parents=True,exist_ok=True)\n        Image.fromarray(np.asarray(arr,dtype=np.uint8)).save(path/\'render_96x72.png\')\n        stage(\'camera_render_ok\',shape=list(np.asarray(arr).shape))\n    stage(\'pass\')\n\nif __name__==\'__main__\':\n    try:main()\n    except BaseException as exc:\n        stage(\'exception\',exception_type=type(exc).__name__,reason=str(exc))\n        traceback.print_exc()\n        raise\n', encoding = 'utf8')
import ast
ast.parse(PROBE.read_text())
print('probe created:', PROBE.resolve())

probe created: /content/probe06.py


In [4]:
import os
import subprocess
import sys
import time
import signal
import json
import zipfile
from pathlib import Path
BACKEND = 'cuda'  # CUDAを確認します。CPUとの比較時のみ 'cpu'
GL_BACKEND = 'egl' # まず egl。描画だけ失敗したら osmesa を別ランタイムで検証
STAGES = ['torch','genesis_import','genesis_init','physics','render']
LIMITS = {'torch':75,'genesis_import':120,'genesis_init':240,'physics':360,'render':300}
results = []
for stage in STAGES:
    name=f'{stage}_{BACKEND}_{GL_BACKEND}'
    dest=OUT/f'{name}.log'
    args = [sys.executable,'-u', str(PROBE),'--test', stage,'--backend', BACKEND,'--gl', GL_BACKEND,'--output', str(OUT)]
    print('RUNNING', stage,'timeout=', LIMITS[stage], flush = True)
    start = time.monotonic()
    with dest.open('w', encoding = 'utf8') as f:
        try:
            # 全標準出力をディスクへ即時出力。Jupyterへ大量の画像・ログを載せない。
            proc = subprocess.Popen(args, stdout = f, stderr = subprocess.STDOUT, start_new_session = True)
            try:
                rc = proc.wait(timeout = LIMITS[stage])
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                proc.wait()
                rc = 124
                f.write('\nDIAG_TIMEOUT\n')
        except Exception as err:
            rc = 125
            f.write(f'\nPARENT_ERROR {type(err).__name__}: {err}\n')
    last = '\n'.join(dest.read_text(errors = 'replace').splitlines()[-10:])
    results.append({'stage':stage,'returncode':rc,'elapsed_s':round(time.monotonic()-start, 1),'log':str(dest)})
    (OUT/'results.json').write_text(json.dumps(results, ensure_ascii = False, indent = 2))
    print('DONE', stage,'returncode=', rc,'elapsed_s=', results[-1]['elapsed_s'])
    print(last[-2000:])
    if rc != 0:
        print('STOP: 以下のステージは未実行です。最初の失敗ログを共有してください。')
        break
print('総合診断:', OUT/'results.json')

RUNNING torch timeout= 75
DONE torch returncode= 1 elapsed_s= 7.4
                                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: torch.cuda.is_available=False
Traceback (most recent call last):
  File "/content/probe06.py", line 65, in <module>
    try:main()
        ~~~~^^
  File "/content/probe06.py", line 23, in main
    if not torch.cuda.is_available():raise RuntimeError('torch.cuda.is_available=False')
                                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: torch.cuda.is_available=False
STOP: 以下のステージは未実行です。最初の失敗ログを共有してください。
総合診断: /content/genesis06_diagnostics/results.json


## D. 診断ログZIPをダウンロード

In [5]:
from pathlib import Path
import zipfile
zip_path = Path('genesis06_diagnostics.zip')
with zipfile.ZipFile(zip_path,'w', compression = zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*'):
        if p.is_file():
            z.write(p, arcname = p.name)
print('診断ZIP:', zip_path.resolve(), round(zip_path.stat().st_size/1024, 1),'KB')
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Colabでない場合は上記ZIPをダウンロードしてください')

診断ZIP: /content/genesis06_diagnostics.zip 1.5 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## E. 結果の読み方

| 最初の失敗 | 主要な切り分け先 |
|---|---|
| torch | GPU割当・PyTorch CUDA・ドライバ |
| genesis_import | 依存ライブラリ・ABI・Python環境 |
| genesis_init | 物理バックエンド初期化・CUDA/JIT |
| physics | Scene.build / JIT / solver / デバイスメモリ |
| render | EGL/OpenGLドライバ・オフスクリーン描画 |

- `returncode=-11` はSIGSEGV（ネイティブ領域のクラッシュ）を示しますが、どのライブラリかはログと突合します。
- `returncode=-9` はSIGKILLであり、メモリ不足の可能性はありますが断定できません。
- 124はこの診断Notebook側が時間超過を検出して停止させたことを示します。
- `RuntimeError: Event loop is closed` はJupyterカーネル再起動に伴う後続のログになりうるため、直前の原因とは限りません。

**重要：** 3D描画が失敗し物理演算が成功した場合、Genesisを物理シミュレータとして残し、別の描画パイプラインでMP4を生成する方向に切り替えられます。物理計算から失敗した場合、課題締切を考慮してMuJoCoの公式Colab実装を代替候補とします。